In [3]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

input_folder = Path('./data')
output_folder = Path('./data_jpg')

records = []  # to build a metadata dataframe alongside the images

for file_path in sorted(input_folder.glob('*.mat')):
    with h5py.File(file_path, 'r') as f:
        image_data = f['cjdata/image'][:]
        label_data = f['cjdata/label'][:]
        pid_data = f['cjdata/PID'][:]
        tumorMask = f['cjdata/tumorMask'][:]

        image_data = image_data.T

        label = int(label_data[0, 0])
        tumor_area = tumorMask.sum()

        # PID is stored as a MATLAB char array -> HDF5 gives you character codes, not digits
        pid_str = ''.join(chr(int(c)) for c in pid_data.flatten())

        im1 = image_data.astype(np.float64)
        min1 = im1.min()
        max1 = im1.max()
        im = (255.0 / (max1 - min1) * (im1 - min1)).astype(np.uint8)

        label_folder = output_folder / str(label)
        label_folder.mkdir(parents=True, exist_ok=True)

        filename = f"{file_path.stem}.jpg"
        output_file_path = label_folder / filename
        Image.fromarray(im).save(output_file_path)

        records.append({
            'filename': filename,
            'label': label - 1,  # keep this consistent with whatever 0-indexing your dataset code expects
            'PID': pid_str,
            'source_file': file_path.stem,
            'tumor_area': tumor_area
        })

metadata_df = pd.DataFrame(records)
metadata_df.to_csv('brain_tumor_metadata.csv', index=False)
print(metadata_df.head())
print(f"\nTotal images: {len(metadata_df)}, Unique patients: {metadata_df['PID'].nunique()}")

   filename  label     PID source_file  tumor_area
0     1.jpg      0  100360           1        4569
1    10.jpg      0  101016          10        3707
2   100.jpg      0  107494         100        2443
3  1000.jpg      2  112649        1000        1542
4  1001.jpg      2  112649        1001        1757

Total images: 3064, Unique patients: 233
